# 02 — Feature Engineering

Builds the two features the model actually consumes, on top of the cleaned dataset from `01_cleaning.ipynb`:
- `embedding` — the text blob that gets vectorized by SBERT
- `quality_score` — a composite "is this actually a good game" signal, used later to re-rank raw semantic matches

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_pickle('../game_on/data/df_clean_step1.pkl')
df.shape

## 2. Building the text blob used for embeddings

Each field gets a short label prefix (`"Game genre: ..."`) so the embedding model has more context than raw comma-separated values.

In [ ]:
df['review_percentage'] = df['review_per'].apply(lambda x: f"Percentage of player recommendations: {int(x)}%" if pd.notna(x) else x)
df['popular_tags'] = df['popular_tags'].apply(lambda x: f"Popular tags: {x}" if pd.notna(x) else x)
df['game_details'] = df['game_details'].apply(lambda x: f"Game details: {x}" if pd.notna(x) else x)
df['genre'] = df['genre'].apply(lambda x: f"Game genre: {x}" if pd.notna(x) else x)

df['embedding'] = (
    df['name'] + '\n' +
    df['genre'] + '\n' +
    df['popular_tags'] + '\n' +
    df['game_details'] + '\n' +
    df['review_percentage'] + '\n' +
    df['required_age'] + '\n' +
    df['game_description']
)

df[['name', 'embedding']].iloc[0]['embedding']

## 3. Quality score

Blends three signals, each only used when available for a given game:
- review percentage (30%)
- log-normalized review count (50%) — so a game with 100k reviews doesn't completely drown out one with 2k
- Metacritic score (20%)

In [ ]:
def calcular_quality_score(row, max_reviews):
    scores = []
    pesos = []

    if pd.notna(row['review_per']):
        scores.append(row['review_per'] / 100)
        pesos.append(0.30)

    if pd.notna(row['total_review']) and row['total_review'] > 0:
        norm = np.log1p(row['total_review']) / np.log1p(max_reviews)
        scores.append(norm)
        pesos.append(0.50)

    if pd.notna(row['metacritic_score']):
        scores.append(row['metacritic_score'] / 100)
        pesos.append(0.20)

    if not scores:
        return None

    total_pesos = sum(pesos)
    return round(sum(s * p for s, p in zip(scores, pesos)) / total_pesos, 4)

max_reviews = df['total_review'].max()
df['quality_score'] = df.apply(lambda row: calcular_quality_score(row, max_reviews), axis=1)

df['quality_score'].describe()

## 4. Save the featured dataset

This is the exact file `game_on/nlp_model` and the API load — `03_EDA.ipynb` and `04_modeling.ipynb` both start from here.

In [ ]:
df = df.dropna(subset=['embedding'])
df.to_pickle('../game_on/data/df_clean.pkl')
print(f"Saved {len(df)} games with embedding text + quality_score")